# 01 Governance

Own the complete Governance lifecycle in one persistent notebook: establish steward and agreement context before engineering, then return after `02_pipeline` produces catalogue and profile evidence to select the governed `table_id`, author Enrichment and Guardrails in one Data Contract editor, review the definition, and freeze its immutable version.

Required delivery flow: **Governance → Engineering → Governance** (`01_governance` → `02_pipeline` → `01_governance`). Optional support remains in `99_explore`.

FabricOps uses standalone widget cells because smaller widget outputs are more stable in Microsoft Fabric notebooks.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release  | Tested by | Date tested | 
|---|---|---| 
|-|  - | -  | 


## 1. Run `00_env_config`

In [ ]:
%run 00_env_config

## 2. Import required Governance functions

In [ ]:
from fabricops_kit import (
    widget_activate_data_contract,
    widget_author_data_contract,
    widget_render_data_agreement,
    widget_render_data_steward,
    widget_view_catalogue,
)


## 3. Data Steward

Run this standalone cell to create or update Data Steward metadata. Run it before establishing a Data Agreement so the accountable parties are available for selection.

In [ ]:
steward_widget = widget_render_data_steward(spark=spark)

## 4. Data Agreement

Create or select the overarching governance agreement between accountable producer and consumer stewards. The agreement defines purpose, scope, ownership, permitted use, and governance conditions. Run this standalone cell after at least one active steward exists.

In [ ]:
agreement_widget = widget_render_data_agreement(spark=spark)

## 5. Select the governed table

After `02_pipeline` has registered and profiled the target, return here and select its canonical `table_id`. Governance reads `METADATA_DATA_CATALOGUE` and `METADATA_DATA_PROFILED` from the metadata target configured by `00_env_config`.


In [ ]:
catalogue_widget = widget_view_catalogue(
    mode="explore",
    target="metadata",
    spark_session=spark,
)
table_selection = catalogue_widget["get_selection"]()
TABLE_ID = table_selection["table_id"]


In [ ]:
views = catalogue_widget["get_views"]()
catalogue_df = views["catalogue"]
profile_df = views["profile"]
frequency_df = views["frequency"]
display(catalogue_df)
display(profile_df)
display(frequency_df)


## 6. Author and freeze the Data Contract

Open the exact draft for `TABLE_ID` in the unified editor. Use its **Enrichment** section for descriptive metadata, **Guardrails** for enforced requirements, and **Review** to inspect the complete table-centric definition before freezing it.

At this stage, the Data Contract binds Enrichment and Guardrails to one governed `table_id`. Data Agreement linkage is finalized later, with the separate Production activation decision. Selection in `02_pipeline` and activation do not belong in this authoring step.


The unified editor creates or reopens the agreement-free draft for `TABLE_ID`.


In [ ]:
contract_authoring = widget_author_data_contract(
    table_id=TABLE_ID,
    spark_session=spark,
)
assert contract_authoring["table_id"] == TABLE_ID


## 7. Review and freeze

In the unified editor, open **Review**, confirm the selected `table_id`, Enrichment, Guardrails, schema, and processing definition, then use the editor's freeze action to save the immutable Data Contract version.

Freezing does not select the version in Engineering Development, activate it for Production, promote `02_pipeline`, or finalize its Data Agreement linkage.


## 8. Link the Data Agreement and activate

After Engineering validates the frozen version in `02_pipeline`, return here. Select the tested table/version and the exact Data Agreement version, review the linkage, and activate it for Production. This is separate from Step 3 authoring and does not promote the pipeline.


In [ ]:
activation = widget_activate_data_contract(
    table_id=TABLE_ID,
    spark_session=spark,
)
